System construction and test


In [1]:
from datetime import date, datetime
import pandas as pd
#import yfinance as yf
import time
import numpy as np
pd.options.mode.chained_assignment = None  # default='warn'
from itertools import product
import sqlite3


input:
    
     Titulo  : example: GGAL
     frequencia de tick : 1h (frequencia mais alta)
     dftitulo : vista do BD sqtitulosalpha.bd (testar ultimos periodas)


#define back testing
sqtitulos() , tbtitulo, symbol, intervalo, dataini, datafim

sqbacktesting
trading system , parameters , parameters range (max , min) , parameters steps
stoploss, stpl parameters, stpl parameters range , stpl parameters steps
stopdrawdown , stdr parameters(drawmax).max.min.step
index, indx parameters (comissions) 


In [5]:
id_backtest = 1

In [7]:
import sqlite3
import pandas as pd

# Conecta ao banco de dados SQLite
con = sqlite3.connect('sqTradeSys.db')  # ou o caminho correto do seu arquivo .sqlite

# Consulta SQL para extrair o registro
query = f"SELECT * FROM vwbacktest WHERE id_backtest = {id_backtest}"

# Executa a consulta e lê em um DataFrame
df = pd.read_sql_query(query, con)

# Converte o primeiro (e único) registro em Series
srbacktest = df.iloc[0] if not df.empty else None

# Fecha a conexão (opcional)
con.close()
display (srbacktest)

id_backtest                                                    1
id_titulos                                                     1
dataini                                      2020-04-03 19:30:00
datafim                                      2024-04-03 19:30:00
symbol                                                      GGAL
intervalo                                                  60min
moeda                                                        USD
source         C:\Users\scitr\anaconda_projects\Trading_Syste...
Name: 0, dtype: object

In [9]:
dataini = srbacktest['dataini']
datafim =  srbacktest['datafim']
display (dataini , datafim)

'2020-04-03 19:30:00'

'2024-04-03 19:30:00'

In [11]:
#%%timeit
import sqlite3
import pandas as pd

# Caminho para o banco de dados

caminho_bd =  srbacktest['source'] 
# Conectando ao banco
conexao = sqlite3.connect(caminho_bd)

# Lendo a view
#consulta = 'SELECT * FROM vwtitulosdados ORDER BY datetime'
consulta = f"""
SELECT * FROM vwtitulosdados
WHERE datetime BETWEEN '{dataini}' AND '{datafim}' AND symbol = '{srbacktest['symbol']}' AND intervalo = '{srbacktest['intervalo']}' AND moeda = '{srbacktest['moeda']}'
ORDER BY datetime
"""

dftitulosdados = pd.read_sql_query(consulta, conexao)

# Fechando a conexão
conexao.close()

# Exibindo os primeiros registros para conferir
#display(dftitulosdados)
dftitulosdados = dftitulosdados.drop(columns=["symbol", "moeda", "intervalo"])
display(len(dftitulosdados))
display(dftitulosdados.head(10))

8957

,datetime,open,high,low,close,volume
0,2020-04-06 08:00:00,5.8034,5.8034,5.8034,5.8034,100.0
1,2020-04-06 09:00:00,6.0432,6.2510,5.9952,6.0832,114582.0
2,2020-04-06 10:00:00,6.0945,6.0945,5.6635,5.7954,159266.0
3,2020-04-06 11:00:00,5.7954,5.8034,5.4837,5.6835,242263.0
4,2020-04-06 12:00:00,5.6715,5.8354,5.6355,5.8274,186504.0
5,2020-04-06 13:00:00,5.8274,5.8354,5.7474,5.7834,68220.0
6,2020-04-06 14:00:00,5.7714,5.7794,5.6435,5.6435,84043.0
7,2020-04-06 15:00:00,5.6595,5.6595,5.5316,5.5956,122920.0
8,2020-04-06 16:00:00,5.6036,5.6036,5.6036,5.6036,49293.0
9,2020-04-07 09:00:00,5.9793,6.0672,5.7395,6.0032,101621.0


In [13]:
# Conecta ao banco de dados SQLite
con = sqlite3.connect('sqTradeSys.db')  # ou o caminho correto do seu arquivo .sqlite

# Consulta SQL para extrair o registro
#query = f"SELECT * FROM vwbacktestparameters ,name, max, min,step WHERE id_backtest = {id_backtest}"
query = f"SELECT  name, type, max, min, step FROM vwbacktestparameters WHERE id_backtest = {id_backtest}"
# Executa a consulta e lê em um DataFrame
df = pd.read_sql_query(query, con)

# Fecha a conexão (opcional)
con.close()

display (df)

,name,type,max,min,step
0,K,int,20,10,2
1,D,int,15,5,3
2,smoth,int,12,4,3
3,medM,int,10,4,2
4,lowM,int,10,4,2
5,stpl,float,0.04,0.01,0.01
6,comission,float,0.006,0.000,0.002
7,drawmax,float,0.2,0.1,0.1


In [15]:
import pandas as pd
import numpy as np
from itertools import product

def parameters_combinator(dfcomb: pd.DataFrame) -> pd.DataFrame:
    """
    Gera um DataFrame com todas as combinações possíveis de parâmetros
    definidos em dfcomb, respeitando os tipos especificados.

    Parâmetros esperados em dfcomb:
    - name: nome da coluna
    - type: tipo de dado ('int' ou 'float')
    - min: valor mínimo
    - max: valor máximo
    - step: incremento

    Retorna:
    - dfparamtest: DataFrame com todas as combinações possíveis
    """
    param_ranges = {}

    for _, row in dfcomb.iterrows():
        name = row['name']
        tipo = row['type']

        # Converte min, max, step para o tipo correto
        if tipo == 'int':
            min_val = int(row['min'])
            max_val = int(row['max'])
            step_val = int(row['step'])
        elif tipo == 'float':
            min_val = float(row['min'])
            max_val = float(row['max'])
            step_val = float(row['step'])
        else:
            raise ValueError(f"Tipo não suportado: {tipo}")

        # Gera a faixa de valores
        values = np.round(np.arange(min_val, max_val + step_val, step_val), 5)
        param_ranges[name] = values

    # Gera todas as combinações possíveis
    combinations = list(product(*param_ranges.values()))

    # Cria o novo DataFrame
    dfparamtest = pd.DataFrame(combinations, columns=param_ranges.keys())

    # Aplica os tipos definidos
    for _, row in dfcomb.iterrows():
        col = row['name']
        tipo = row['type']
        if tipo == 'int':
            dfparamtest[col] = dfparamtest[col].astype(int)
        elif tipo == 'float':
            dfparamtest[col] = dfparamtest[col].astype(float)

    return dfparamtest
dfcomb = df

In [17]:
dfcomb = df
dfparamtest = parameters_combinator(dfcomb)
print(dfparamtest.head())
print(len(dfparamtest))

    K  D  smoth  medM  lowM  stpl  comission  drawmax
0  10  5      4     4     4  0.01      0.000      0.1
1  10  5      4     4     4  0.01      0.000      0.2
2  10  5      4     4     4  0.01      0.000      0.3
3  10  5      4     4     4  0.01      0.002      0.1
4  10  5      4     4     4  0.01      0.002      0.2
92160


In [19]:
dfmetricas = None

START LOOP

In [22]:
il = 70000

In [24]:
# Parameters

# System parameters
# stoch_hml_1
K = dfparamtest.loc[il,'K']
D = dfparamtest.loc[il,'D']
smoth = dfparamtest.loc[il,'smoth']
medM = (dfparamtest.loc[il,'medM'])
lowM = (dfparamtest.loc[il,'lowM'])

# Backtesting parameters

stpl = dfparamtest.loc[il,'stpl']
comission = dfparamtest.loc[il,'comission']
drawmax = dfparamtest.loc[il,'drawmax']
print(K,D, smoth, stpl, drawmax)

18 11 13 0.02 0.2


In [26]:


lsmetricas = []
srparamtest = dfparamtest.loc[il]
lsmetricas.append(srparamtest)

print(srparamtest)


K            18.000
D            11.000
smoth        13.000
medM          4.000
lowM          8.000
stpl          0.020
comission     0.002
drawmax       0.200
Name: 70000, dtype: float64


Trading System

In [103]:
# Trading system: Stoch_HighMedLow_Long

import numpy as np
import pandas as pd

# Stochastic calculation
def stochastic(dftitulosdados, i, K, D, smoth):
    df = dftitulosdados.copy()
    df["k"] = (100. * (df.close - df.low.rolling(K).min()) /
               (df.high.rolling(K).max() - df.low.rolling(K).min()))
    
    df["k" + i] = df["k"].rolling(smoth).mean()
    df["d" + i] = df["k" + i].rolling(D).mean()
    
    df.drop(columns=["k"], inplace=True)
    return df

# Stochastic high, med and low frequency
def stoch_hml(dfstoch, k, d, smth, medM, lowM):
    df = dfstoch.copy()
    df = stochastic(df, "high", k, d, smth)
    df = stochastic(df, "med", k * medM, d * medM, smth * medM)
    df = stochastic(df, "low", k * medM * lowM, d * medM * lowM, smth * medM * lowM)
    return df

# Criteria calculation
def system_criterias(dfstoch_hml):
    df = dfstoch_hml.copy()
    df["longbuylow"] = ((df["klow"] > 20) & (df["klow"] > df["dlow"])).astype(int)
    df["longbuymed"] = ((df["kmed"] > 20) & (df["kmed"] > df["dmed"])).astype(int)
    df["longbuyhigh"] = ((df["khigh"] > 20) & (df["khigh"] > df["dhigh"])).astype(int)
    return df

# Signal generation
def system_signals(dfcriterias):
    df = dfcriterias.copy()
    n = len(df)
    state_array = np.full(n, "standby", dtype=object)
    estado_anterior = "standby"

    high = df["longbuyhigh"].to_numpy()
    med = df["longbuymed"].to_numpy()
    low = df["longbuylow"].to_numpy()

    for i in range(1, n):
        if high[i] == 1 and med[i] == 1 and low[i] == 1 and estado_anterior == "standby":
            state_array[i] = "enter"
            estado_anterior = "enter"
        elif med[i] == 1 and low[i] == 1 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "stay"
            estado_anterior = "stay"
        elif high[i] == 1 and low[i] == 1 and med[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "stay"
            estado_anterior = "stay"
        elif low[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "out"
            estado_anterior = "out"
        elif high[i] == 0 and med[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "out"
            estado_anterior = "out"
        elif (low[i] == 0 or med[i] == 0) and estado_anterior == "out":
            state_array[i] = "standby"
            estado_anterior = "standby"
        else:
            state_array[i] = estado_anterior

    df["state"] = state_array
    dfsignals = df.drop(columns=[
        "khigh", "dhigh", "kmed", "dmed", "klow", "dlow",
        "longbuylow", "longbuymed", "longbuyhigh"
    ])
    return dfsignals

def Stoch_HighMedLow_Long (dftitulosdados, K, D, smoth, medM , lowM):     #Columns: datetime,open,high,low,close,volume 
    dfstoch = stochastic(dftitulosdados, 'high', K, D, smoth)
    dfstoch_hml= stoch_hml(dfstoch , K, D, smoth, medM , lowM )
    dfcriterias = system_criterias (dfstoch_hml)
    dfsignals = system_signals (dfcriterias)
    return dfsignals                                                      #Columns: datetime,open,high,low,close,volume,state

In [105]:
dfsignals = Stoch_HighMedLow_Long (dftitulosdados, K, D, smoth, medM , lowM)

In [111]:
                                 #stop_loss_reentry

def stop_loss_reentry (dfsignals, stpl) :                 #Columns: datetime,open,high,low,close,volume,state

    df = dfsignals[(dfsignals['state'] == 'enter') | (dfsignals['state'] == 'stay')]
    df = df.reset_index(drop=True)
    
    
    df["stpl"] = 0.0
    stoplossprice = 0.0
    lastlongbuyprice = 0.0

    for i in range(0, len(df)):
        
        if df.loc[i, "state"] == "enter" :
           stoplossprice = df.loc[i, "close"]
           df.loc[i,"stpl"] = df.loc[i, "close"] - stoplossprice * (1 - stpl)
           
        if df.loc[i, "state"]== "stay" :
           df.loc[i, "stpl"] = df.loc[i, "close"] - stoplossprice * (1- stpl)
            
           if df.loc[i, "stpl"] < 0.0 :
                df.loc[i, "state"] = "out"
               
           if df.loc[i, "stpl"] > 0.0 and   (df.loc[i-1, "state"] == "out" or df.loc[i-1, "state"] == "outstpl") :
                df.loc[i, "state"] = "enter"
               
           if df.loc[i, "stpl"] < 0.0 and   (df.loc[i-1, "state"] == "out" or df.loc[i-1, "state"] == "outstpl") :
                df.loc[i, "state"] = "outstpl"
    dfstoploss = df

    # reensablar e limpar duplicados out dfsignals 
    dfsignalsout = dfsignals[(dfsignals['state'] == 'out')]
     # elimino a culuna stpl de dfstoploss e filtro os valores enter e out 
    dfstoplossdrop = dfstoploss.drop(columns=["stpl"]) 
    dfstoplossenterout = dfstoplossdrop[(dfstoplossdrop['state'] == 'enter') | (dfstoplossdrop['state'] == 'out')]

    # concatenar os dois df para ter o total dos signals enter e out
    dfsignalsenterout = pd.concat([dfsignalsout, dfstoplossenterout], ignore_index=True) 
    # Ordenar pelo datetime e resetear o index
    dfsignalsenterout["datetime"] = pd.to_datetime(dfsignalsenterout["datetime"])
    dfsignalsenterout = dfsignalsenterout.sort_values("datetime").reset_index(drop=True)

    # limpar os out duplicados"out" por a saida anticipada do stoploss e reiniciar indice
    df = dfsignalsenterout
    cond = (df["state"] == "out")  & (df["state"].shift(1) == "out")
    dfsignals = df[~cond].reset_index(drop=True)
    
    return dfsignals , dfstoploss                            #Columns: datetime,open,high,low,close,volume,state,stpl


In [113]:
dfsignals, dfstoploss = stop_loss_reentry (dfsignals, stpl)
display (dfsignals)
#display (dfstoploss)


,datetime,open,high,low,close,volume,state
0,2020-12-01 18:00:00,7.2376,7.2376,7.2376,7.2376,100.0,enter
1,2020-12-08 09:00:00,7.2780,7.3266,7.2214,7.2376,21139.0,out
2,2020-12-29 11:00:00,7.2133,7.4074,7.2133,7.3993,226749.0,enter
3,2020-12-30 13:00:00,7.3063,7.3185,7.2376,7.2484,87356.0,out
4,2021-01-20 13:00:00,6.3966,6.4046,6.3400,6.3965,51857.0,enter
...,...,...,...,...,...,...,...
127,2024-02-28 07:00:00,19.6522,19.6522,19.6522,19.6522,96.0,out
128,2024-03-12 12:00:00,21.1867,21.2699,20.9833,21.0757,127082.0,enter
129,2024-03-12 17:00:00,20.3104,20.3104,20.3104,20.3104,2.0,out
130,2024-03-12 19:00:00,21.4362,21.4362,21.4362,21.4362,50.0,enter


In [119]:
#Index_sc, Index, Trade calculation
# Indexes calculation
def index_calculation (dfsignals):   # Columns datetime	open	high	low	close	volume	state
    df = dfsignals
    df ["index_sc"] = 100. 
    df ["trade"] = 0.
    df ["index"] = 100. *(1-comission) 
    for i in range(1, len(df)):      
                       
        if  df.loc[i, "state"] == "out" :
            df.loc[i, "index_sc"] = (((df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"])+1)* df.loc[i-1,"index_sc"]
            df.loc[i, "index"] = df.loc[i, "index_sc"]* (1-comission)
            df.loc[i, "trade"] = (df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"]
            
        if  df.loc[i, "state"] == "enter" :        
            df.loc[i, "index_sc"] =  df.loc[i-1, "index_sc"]
            df.loc[i, "index"] = df.loc[i, "index_sc"]* (1-comission)
    dfindex = df
    return dfindex    #Columns : datetime	open	high	low	close	volume	state	index_sc	trade	index
    


In [121]:
dfindex = index_calculation (dfsignals)
display (dfindex)


,datetime,open,high,low,close,volume,state,index_sc,trade,index
0,2020-12-01 18:00:00,7.2376,7.2376,7.2376,7.2376,100.0,enter,100.000000,0.000000,99.800000
1,2020-12-08 09:00:00,7.2780,7.3266,7.2214,7.2376,21139.0,out,100.000000,0.000000,99.800000
2,2020-12-29 11:00:00,7.2133,7.4074,7.2133,7.3993,226749.0,enter,100.000000,0.000000,99.800000
3,2020-12-30 13:00:00,7.3063,7.3185,7.2376,7.2484,87356.0,out,97.960618,-0.020394,97.764697
4,2021-01-20 13:00:00,6.3966,6.4046,6.3400,6.3965,51857.0,enter,97.960618,0.000000,97.764697
...,...,...,...,...,...,...,...,...,...,...
127,2024-02-28 07:00:00,19.6522,19.6522,19.6522,19.6522,96.0,out,69.929060,0.071035,69.789202
128,2024-03-12 12:00:00,21.1867,21.2699,20.9833,21.0757,127082.0,enter,69.929060,0.000000,69.789202
129,2024-03-12 17:00:00,20.3104,20.3104,20.3104,20.3104,2.0,out,67.389799,-0.036312,67.255019
130,2024-03-12 19:00:00,21.4362,21.4362,21.4362,21.4362,50.0,enter,67.389799,0.000000,67.255019


In [131]:
# Stop System Calculation , "stopsys" asignation into state column

def stop_drawdown (dfindex, drawmax):
    df = dfindex
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df["index"] = pd.to_numeric(df["index"], errors="coerce")

    estado_corrigido = []
    pico_atual = df.loc[0, "index"]

    for i in range(len(df)):
        valor_index = df.loc[i, "index"]
        estado = df.loc[i, "state"]

        # Atualiza pico se houve recuperação
        if valor_index > pico_atual:
            pico_atual = valor_index

        # Calcula drawdown
        if pico_atual > 0:
            drawdown = (valor_index - pico_atual) / pico_atual
        else:
            drawdown = 0

        # Verifica se deve aplicar stopsys
        if estado == "out" and drawdown < (- drawmax):
            estado = "stopsys"
            pico_atual = valor_index  # reinicia ciclo a partir desse ponto

        estado_corrigido.append(estado)
    df["state"] = estado_corrigido
    dfindexdrawdown = df
    return dfindexdrawdown



In [133]:
dfindexdrawdown = stop_drawdown(dfindex, drawmax)
display(dfindexdrawdown)

,datetime,open,high,low,close,volume,state,index_sc,trade,index
0,2020-12-01 18:00:00,7.2376,7.2376,7.2376,7.2376,100.0,enter,100.000000,0.000000,99.800000
1,2020-12-08 09:00:00,7.2780,7.3266,7.2214,7.2376,21139.0,out,100.000000,0.000000,99.800000
2,2020-12-29 11:00:00,7.2133,7.4074,7.2133,7.3993,226749.0,enter,100.000000,0.000000,99.800000
3,2020-12-30 13:00:00,7.3063,7.3185,7.2376,7.2484,87356.0,out,97.960618,-0.020394,97.764697
4,2021-01-20 13:00:00,6.3966,6.4046,6.3400,6.3965,51857.0,enter,97.960618,0.000000,97.764697
5,2021-01-21 10:00:00,6.2753,6.2834,6.1297,6.1753,133930.0,out,94.573001,-0.034581,94.383855
6,2021-01-27 12:00:00,6.1297,6.2510,6.1297,6.2268,97839.0,enter,94.573001,0.000000,94.383855
7,2021-01-29 13:00:00,6.1459,6.1782,6.0731,6.0732,45168.0,out,92.240116,-0.024668,92.055635
8,2021-02-01 08:00:00,6.1459,6.1459,6.1459,6.1459,500.0,enter,92.240116,0.000000,92.055635
9,2021-02-17 15:00:00,6.9060,7.0597,6.9060,6.9748,115008.0,out,104.680577,0.134870,104.471216


In [127]:
# METRICS
# creo dataframe para calculo de metricas
dfinputmetricas = dfindex[['datetime', 'state','index_sc', 'index','trade']]
#Metricas list inicialicion 

pd.set_option('display.max_rows', None)

#display(dfinputmetricas, lsmetricas)

In [68]:
# Tir Total Metric
# Tir total calculation
def tir_total_anualizada(dfinputmetricas, lsmetricas):
    df = dfinputmetricas
    # Garante que datetime está no formato certo
    df['datetime'] = pd.to_datetime(df['datetime'])

    # Filtra enter e out
    df_enter = df[df['state'] == 'enter']
    df_out = df[df['state'] == 'out']

    # Verificação
    if df_enter.empty or df_out.empty:
        return None

    # Índice inicial e final
    idx_inicio = df_enter.iloc[0]['index']
    idx_fim = df_out.iloc[-1]['index']

    # Período completo entre primeira e última data do DataFrame
    dt_inicio_total = df['datetime'].min()
    dt_fim_total = df['datetime'].max()
    dias_total = (dt_fim_total - dt_inicio_total).days

    # Validação
    if dias_total <= 0 or idx_inicio == 0:
        return None

    # TIR anualizada com base no período total do df
    tirtotalanual = (idx_fim / idx_inicio) ** (365 / dias_total) - 1
    setirtotalanual = pd.Series({'tirtotalanual': tirtotalanual})
    lsmetricas.append (setirtotalanual)
    return  setirtotalanual , lsmetricas

setirtotalanual , lsmetricas = tir_total_anualizada (dfinputmetricas, lsmetricas)
#display (setirtotalanual, lsmetricas)

    

NameError: name 'dfinputmetricas' is not defined

In [ ]:
#%%timeit
#Tir Anuais Metrics
# dataframe  calculation
def tir_anuais_df (dfinputmetricas, dataini, datafim):

    # Exemplo do DataFrame original
    df = dfinputmetricas
    dataini = pd.to_datetime(dataini)
    datafim = pd.to_datetime(datafim)
    
    # Lista para novos registros
    novos_registros = []
    
    # Verifica se dataini deve ser adicionado
    if dataini < df.iloc[0]['datetime']:
        novos_registros.append({
            'datetime': dataini,
            'state': '',
            'index': df.iloc[0]['index']
        })
    
    # Verifica se datafim deve ser adicionado
    if datafim > df.iloc[-1]['datetime']:
        novos_registros.append({
            'datetime': datafim,
            'state': '',
            'index': df.iloc[-1]['index']
        })
    
    # Adiciona os registros e ordena
    df = pd.concat([pd.DataFrame(novos_registros), df], ignore_index=True)
    df = df.sort_values(by='datetime').reset_index(drop=True)
    
    # Determina os anos, excluindo o último ano
    ano_inicial = df['datetime'].min().year
    ano_final = (df['datetime'].max().year)
    
    # Gera os anos do intervalo EXCLUINDO o último ano
    anos_validos = range(ano_inicial, ano_final-1 )  # << ajuste aqui
    
    # Lista para os novos registros
    novos_registros = []
    
    for ano in anos_validos:
        fim_do_ano = pd.to_datetime(f'{ano}-12-31 23:59:59')
        df_antes = df[df['datetime'] < fim_do_ano]
        if not df_antes.empty:
            index_valor = df_antes.iloc[-1]['index']
            novos_registros.append({
                'datetime': fim_do_ano,
                'state': '',
                'index': index_valor
            })
    
    # Adiciona e organiza
    df = pd.concat([df, pd.DataFrame(novos_registros)], ignore_index=True)
    df = df.sort_values('datetime').reset_index(drop=True)
    
    # calcula o dataframe com as tir anuales ao fim do ano , com os anos incompletos anualizadas
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    #  consolidar por dia e manter o último registro
    df['date'] = df['datetime'].dt.date
    df_diario = df.sort_values('datetime').groupby('date', as_index=False).last()
    
    #  selecionar datas de fim de ano
    df_fim_ano = df_diario[
        (pd.to_datetime(df_diario['date']).dt.month == 12) &
        (pd.to_datetime(df_diario['date']).dt.day == 31)
    ].copy()
    
    #  calcular TIR entre pares de fim de ano
    resultados = []
    
    for i in range(1, len(df_fim_ano)):
        dt_inicio = pd.to_datetime(df_fim_ano.iloc[i - 1]['date'])
        dt_fim = pd.to_datetime(df_fim_ano.iloc[i]['date'])
        idx_inicio = df_fim_ano.iloc[i - 1]['index']
        idx_fim = df_fim_ano.iloc[i]['index']
        dias = (dt_fim - dt_inicio).days
    
        if dias > 0 and idx_inicio != 0:
            tir = (idx_fim / idx_inicio) ** (365 / dias) - 1
            resultados.append({
                'datetime': dt_fim,
                'tiranual': tir
            })
    
    #  adicionar último intervalo incompleto
    if not df_fim_ano.empty:
        dt_inicio = pd.to_datetime(df_fim_ano.iloc[-1]['date'])
        idx_inicio = df_fim_ano.iloc[-1]['index']
        dt_fim = pd.to_datetime(df_diario.iloc[-1]['date'])
        idx_fim = df_diario.iloc[-1]['index']
        dias = (dt_fim - dt_inicio).days
    
        if dias > 0 and idx_inicio != 0:
            tir = (idx_fim / idx_inicio) ** (365 / dias) - 1
            resultados.append({
                'datetime': dt_fim,
                'tiranual': tir
            })

    # criar DataFrame final
    dftiranual = pd.DataFrame(resultados)
    return dftiranual

dftiranual = tir_anuais_df (dfinputmetricas, dataini, datafim)

#estatistic calculation
def tir_anuais_estat(dftiranual, lsmetricas):
    tir = dftiranual['tiranual'].dropna()

    estatisticas = {
        'tiranualquant': tir.count(),
        'tiranualfirst': round(tir.iloc[0], 6),
        'tiranualmedia': round(tir.mean(), 6),
        'tiranualmax': round(tir.max(), 6),
        'tiranualmin': round(tir.min(), 6),
        'tiranualstd': round(tir.std(), 6)
    }
    setiranuaisestats = pd.Series(estatisticas)
    lsmetricas.append (setiranuaisestats)
    return setiranuaisestats , lsmetricas

setiranuaisestats , lsmetricas = tir_anuais_estat(dftiranual, lsmetricas)
#display (dftiranual, lsmetricas)


In [69]:
# TRades Metrics
# trades estatistics calculation
def trades_estatisticas(dfinputmetricas, lsmetricas):
    df = dfinputmetricas.copy()
    trades = df["trade"].dropna()

    positivos = trades[trades > 0]
    negativos = trades[trades < 0]

    # Porcentagem de positivos
    porcentagem_pos = (len(positivos) / len(trades)) if len(trades) > 0 else 0

    ditradesestat = {
        "tradestot": len(trades),
        'tradefirst': round(trades.iloc[1], 6),
        "tradespositpor": round(porcentagem_pos, 6),
        "tradesposmedia": round(positivos.mean(), 6) if not positivos.empty else None,
        "tradesposstd": round(positivos.std(), 6) if not positivos.empty else None,
        "tradesposmax": round(positivos.max(), 6) if not positivos.empty else None,
        "tradesposmin": round(positivos.min(), 6) if not positivos.empty else None
    }

    setradesestats = pd.Series(ditradesestat)
    lsmetricas.append (setradesestats)
    return setradesestats , lsmetricas

setradesestats, lsmetricas = trades_estatisticas(dfinputmetricas, lsmetricas)
#display (lsmetricas)

NameError: name 'dfinputmetricas' is not defined

In [583]:
#Drawdown Metrics

#Dataframe Drawdown Calculation
def drawdowns_df (dfinputmetricas):
   
    df = dfinputmetricas
    df["datetime"] = pd.to_datetime(df["datetime"])
    serie = df["index"].dropna().reset_index(drop=True)
    datas = df["datetime"].reset_index(drop=True)

    drawdowns = []

    pico_idx = 0
    pico = serie[0]
    vale_idx = None
    valor_vale = None
    max_dd = 0

    for i in range(1, len(serie)):
        if serie[i] > pico:
            # Se recuperou acima do último pico: salvar ciclo anterior
            if vale_idx is not None and max_dd < 0:
                drawdowns.append({
                    "Data Pico": datas[pico_idx],
                    "Valor Pico": pico,
                    "Data Vale": datas[vale_idx],
                    "Valor Vale": valor_vale,
                    "Drawdown (%)": round(max_dd * 100, 2)
                })

            # Novo pico inicia novo ciclo
            pico = serie[i]
            pico_idx = i
            vale_idx = None
            max_dd = 0
        else:
            dd = (serie[i] - pico) / pico
            if dd < max_dd:
                max_dd = dd
                vale_idx = i
                valor_vale = serie[i]

    # Salva último ciclo, se aplicável
    if vale_idx is not None and max_dd < 0:
        drawdowns.append({
            "Data Pico": datas[pico_idx],
            "Valor Pico": pico,
            "Data Vale": datas[vale_idx],
            "Valor Vale": valor_vale,
            "Drawdown (%)": round(max_dd * 100, 2)
        })

    # Retorna os top N
    df_resultado = pd.DataFrame(drawdowns)
    return df_resultado.sort_values("Drawdown (%)").reset_index(drop=True)  

dfdrawdowns = drawdowns_df(dfinputmetricas)

# Drawdowns statictics calculation
def drawdowns_estat(dfdrawdowns , lsmetricas):
    dd = dfdrawdowns['Drawdown (%)'].dropna()  # Filtra nulos, se houver

    estatisticas = {
        'drawdfirst': round(dd.iloc[0], 6),
        'drawdtot': dd.count(),
        'drawdmedia': round(dd.mean(), 2),
        'drawdmaximo': round(dd.max(), 2),
        'drawdminimo': round(dd.min(), 2),
        'drawdstd': round(dd.std(), 2)
    }
    sedrawdownsestats = pd.Series(estatisticas)
    lsmetricas.append (sedrawdownsestats)
    return sedrawdownsestats , lsmetricas

sedrawdownsestats , lsmetricas = drawdowns_estat(dfdrawdowns, lsmetricas)
#display(lsmetricas)

In [585]:
#Dias Out Metrics

# Dataframe calculation
def dias_out_df (dfmetricas):    
    df = dfmetricas
    coluna="index_sc"
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df[df[coluna].notna()].reset_index(drop=True)

    variacao = df[coluna].diff()
    grupos = (variacao != 0).cumsum()

    agrupado = df.groupby(grupos)
    periodos_estaticos = []

    for _, grupo in agrupado:
        if len(grupo) > 1 and grupo[coluna].nunique() == 1:
            duracao_dias = (grupo["datetime"].iloc[-1] - grupo["datetime"].iloc[0]).days
            periodos_estaticos.append({                
                "Data Início": grupo["datetime"].iloc[0],
                "Data Fim": grupo["datetime"].iloc[-1],
                "difdias": duracao_dias,
                "Valor index": grupo[coluna].iloc[0]
            })

    dfdiasout = pd.DataFrame(periodos_estaticos)
    dfdiasout = dfdiasout.query("difdias != 0").copy()
    dfdiasout = dfdiasout.reset_index(drop=True)
    
    return dfdiasout
    
dfdiasout = dias_out_df (dfinputmetricas)

# Diasout statistic calculation and lsmetricas agregation
def dias_out_estats(dfdiasout, lsmetricas) :
    dias = dfdiasout['difdias'].dropna()  # Remove valores nulos, se houver

    estatisticas = {
        'diasoutfirst': round(dias.iloc[0], 6),
        'diasouttot': dias.sum(),
        'diasoutmedia': round(dias.mean(), 2),
        'diasoutmax': dias.max(),
        'diasoutmin': dias.min(),
        'diasoutstd': round(dias.std(), 2)
    }
    sediasoutestats = pd.Series(estatisticas)
    lsmetricas.append(sediasoutestats)    
    return sediasoutestats, lsmetricas

sediasoutestats, lsmetricas = dias_out_estats(dfdiasout, lsmetricas)
#display(lsmetricas)

In [587]:
#Metrica StopSys

# Dataframe calculation
def stopsys_df (dfindexdrawdown) : 
    df = dfindexdrawdown
    # Garante que a coluna 'datetime' esteja no formato correto
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    # Filtra os registros onde state == 'stopsys'
    dfstopsys = df[df['state'] == 'stopsys'][['datetime', 'state', 'index']].copy()
    
    #  Ordena por datetime
    dfstopsys = dfstopsys.sort_values('datetime').reset_index(drop=True)
    return dfstopsys
    
dfstopsys = stopsys_df (dfindexdrawdown)

# statistic dataframe based calculation and lsmetricas agregation
def stopsys_estat(dfstopsys, lsmetricas):
    df = dfstopsys.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime').reset_index(drop=True)

    index_values = df['index'].dropna()

    if index_values.empty:
        estatisticas = {
            'stopsysfirst': 0.0,
            'stopsysquant': 0,
            'stopsysmedia': 0.0,
            'stopsysmaximo': 0.0,
            'stopsysminimo': 0.0,
            'stopsystd': 0.0
        }
    else:
        estatisticas = {
            'stopsysfirst': round(index_values.iloc[0], 6),
            'stopsysquant': index_values.count(),
            'stopsysmedia': round(index_values.mean(), 6),
            'stopsysmaximo': round(index_values.max(), 6),
            'stopsysminimo': round(index_values.min(), 6),
            'stopsystd': round(index_values.std(), 6)
        }

    sestopsysestats = pd.Series(estatisticas)
    lsmetricas.append(sestopsysestats)
    return sestopsysestats, lsmetricas
sestopsysestats, lsmetricas = stopsys_estat(dfstopsys, lsmetricas)
#display (lsmetricas )

In [589]:


def atualizar_df_metricas(dfmetricas, lsmetricas):
    """
    Adiciona uma linha ao DataFrame dfmetricas com os valores de lsmetricas.
    Se dfmetricas for None, cria o DataFrame com a estrutura das métricas.

    Parâmetros:
    - dfmetricas: pd.DataFrame ou None
    - lsmetricas: list de pd.Series

    Retorna:
    - pd.DataFrame atualizado
    """

    linha = pd.concat(lsmetricas)  # Une todas as Series em uma só

    if dfmetricas is None:
        # Cria o DataFrame com uma única linha
        dfmetricas = pd.DataFrame([linha.values], columns=linha.index)
    else:
        # Adiciona nova linha ao DataFrame existente
        dfmetricas.loc[len(dfmetricas)] = linha.values

    return dfmetricas

dfmetricas = atualizar_df_metricas(dfmetricas, lsmetricas)

In [591]:
display (dfmetricas)

,K,D,smoth,medM,lowM,stpl,comission,drawmax,tirtotalanual,tiranualquant,...,diasoutmedia,diasoutmax,diasoutmin,diasoutstd,stopsysfirst,stopsysquant,stopsysmedia,stopsysmaximo,stopsysminimo,stopsystd
0,20.0,8.0,4.0,4.0,8.0,0.03,0.004,0.3,-0.007930,3.0,...,21.98,109.0,1.0,29.34,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
1,10.0,5.0,7.0,6.0,4.0,0.04,0.002,0.2,-0.040202,3.0,...,22.43,86.0,1.0,23.29,100.348593,4.0,96.734158,105.138853,79.304537,11.786548
2,14.0,17.0,10.0,8.0,6.0,0.03,0.000,0.1,-0.086744,3.0,...,33.38,141.0,1.0,44.01,88.948260,7.0,95.813829,114.630633,73.365784,15.242522
3,18.0,11.0,13.0,4.0,8.0,0.02,0.002,0.2,-0.082787,3.0,...,27.36,103.0,1.0,31.57,82.440911,3.0,67.445087,82.440911,54.062487,14.257826


END LOOP

In [329]:
dfmetricas.insert(
    loc=0,  # insere como primeira coluna
    column="id_backtest",
    value=[srbacktest["id_backtest"]] * len(dfmetricas)
)
display(dfmetricas)

,id_backtest,K,D,smoth,medM,lowM,stpl,comission,drawmax,tirtotalanual,...,diasoutmedia,diasoutmax,diasoutmin,diasoutstd,stopsysfirst,stopsysquant,stopsysmedia,stopsysmaximo,stopsysminimo,stopsystd
0,1,16.0,12.0,6.0,5.0,10.0,0.04,0.005,-0.05,-0.008228,...,25.94,103.0,1.0,30.96,92.631437,18.0,80.080879,105.007263,59.073016,13.316339
